# Spatial Point Patterns

Here the event locations themselves are the random outcome. We simulate a homogeneous Poisson process and a Thomas cluster process, then compare nearest-neighbor distances and a border-corrected estimate of Ripley's \(K\).

For complete spatial randomness in the plane,

\[
K(r)=\pi r^2.
\]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

rng = np.random.default_rng(17)
bounds = (0.0, 1.0, 0.0, 1.0)

def poisson_process(intensity):
    n = rng.poisson(intensity)
    return rng.uniform(0, 1, size=(n,2))

def thomas_process(parent_intensity=8, mean_offspring=12, sd=0.035):
    margin = 4*sd
    area_expanded = (1+2*margin)**2
    n_parents = rng.poisson(parent_intensity*area_expanded)
    parents = rng.uniform(-margin, 1+margin, size=(n_parents,2))
    children = []
    for p in parents:
        m = rng.poisson(mean_offspring)
        if m:
            children.append(p + rng.normal(scale=sd, size=(m,2)))
    pts = np.vstack(children)
    inside = np.all((pts >= 0) & (pts <= 1), axis=1)
    return pts[inside]

csr = poisson_process(100)
cluster = thomas_process()
print(len(csr), len(cluster))


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(9,4))
axes[0].scatter(csr[:,0], csr[:,1], s=12)
axes[0].set_title("Homogeneous Poisson")
axes[1].scatter(cluster[:,0], cluster[:,1], s=12)
axes[1].set_title("Thomas cluster process")
for ax in axes:
    ax.set(xlim=(0,1), ylim=(0,1), aspect="equal")
plt.show()


In [ ]:
def mean_nn(points):
    D = cdist(points, points)
    np.fill_diagonal(D, np.inf)
    return np.min(D, axis=1).mean()

print("mean nearest-neighbor distance")
print("CSR:      ", round(mean_nn(csr), 3))
print("clustered:", round(mean_nn(cluster), 3))


Nearest-neighbor distance is dominated by short-range behavior. Ripley's \(K\) examines a range of distances, but requires edge correction because circles around boundary points extend outside the observed window.

In [ ]:
def boundary_distance(points):
    return np.min(np.column_stack([
        points[:,0], 1-points[:,0], points[:,1], 1-points[:,1]
    ]), axis=1)

def K_border(points, radii):
    n = len(points)
    D = cdist(points, points)
    np.fill_diagonal(D, np.inf)
    b = boundary_distance(points)
    out = []
    for r in radii:
        eligible = b >= r
        m = eligible.sum()
        count = np.sum(D[eligible] <= r)
        out.append(count / (n*m) if m else np.nan)  # area = 1
    return np.array(out)

radii = np.linspace(0.02, 0.18, 25)
K_csr = K_border(csr, radii)
K_cluster = K_border(cluster, radii)

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(radii, np.pi*radii**2, label="CSR theory")
ax.plot(radii, K_csr, label="simulated CSR")
ax.plot(radii, K_cluster, label="cluster process")
ax.set(xlabel="r", ylabel="K(r)", title="Border-corrected Ripley K")
ax.legend()
plt.show()


An observed \(K(r)\) above \(\pi r^2\) is consistent with excess pairs at scale \(r\), but broad-scale variation in intensity can also produce this pattern. First-order intensity and second-order interaction must be separated before attributing clustering to event interaction.